# Experiment 5: CNN Training and Transfer Learning

This notebook studies how a few important training choices affect an image classification model.  
Instead of building a large CNN from scratch, MobileNetV2 is used because it is lightweight and works well for transfer learning.

The experiments cover initialization, regularization, optimizers, hyperparameters, fine-tuning and a small cross-validation study.

## 1. Setup

TensorFlow already provides most of what is needed in Colab. The dataset will be loaded using TensorFlow Datasets.

In [ ]:
!pip install -q --upgrade "protobuf>=5.29.1,<6.0.0"
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

SEED = 42
tf.keras.utils.set_random_seed(SEED)

IMG_SIZE = 224
NUM_CLASSES = 37
AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow:", tf.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 6.8 MB/s eta 0:00:00


ERROR:absl:Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow_metadata/proto/v0/anomalies.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/rlds/envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/co

TensorFlow: 2.20.0


In [ ]:
!pip uninstall -y tensorflow-datasets
!pip install --no-cache-dir tensorflow-datasets==4.9.9

Found existing installation: tensorflow-datasets 4.9.9
Uninstalling tensorflow-datasets-4.9.9:
  Successfully uninstalled tensorflow-datasets-4.9.9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 69.1 MB/s eta 0:00:00


In [ ]:
import tensorflow_datasets as tfds

print("Module:", tfds)
print("File:", getattr(tfds, "__file__", "Unknown"))
print("Version:", getattr(tfds, "__version__", "Unknown"))
print("Has load:", hasattr(tfds, "load"))

ERROR:absl:Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow_metadata/proto/v0/anomalies.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/rlds/envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/co

Module: <module 'tensorflow_datasets' from '/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/__init__.py'>
File: /usr/local/lib/python3.13/dist-packages/tensorflow_datasets/__init__.py
Version: Unknown
Has load: False


## 2. Load and prepare the Oxford-IIIT Pet dataset

The images come in different sizes, so every image is resized to 224 × 224.  
The official training split is divided into training and validation data, while the test split is kept separate until the final evaluation.

In [ ]:
(ds_train_raw, ds_test_raw), ds_info = tfds.load(
    "oxford_iiit_pet",
    split=["train", "test"],
    as_supervised=True,
    with_info=True
)

train_size = ds_info.splits["train"].num_examples
test_size = ds_info.splits["test"].num_examples

print("Training images:", train_size)
print("Test images:", test_size)
print("Classes:", ds_info.features["label"].num_classes)

AttributeError: module 'tensorflow_datasets' has no attribute 'load'

In [ ]:
def resize_image(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, label

all_train = ds_train_raw.map(resize_image, num_parallel_calls=AUTOTUNE)
all_train = all_train.shuffle(train_size, seed=SEED, reshuffle_each_iteration=False)

val_count = int(0.2 * train_size)
val_raw = all_train.take(val_count)
train_raw = all_train.skip(val_count)

test_raw = ds_test_raw.map(resize_image, num_parallel_calls=AUTOTUNE)

def make_dataset(ds, batch_size=32, training=False):
    if training:
        ds = ds.shuffle(1000, seed=SEED)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_dataset(train_raw, 32, training=True)
val_ds = make_dataset(val_raw, 32)
test_ds = make_dataset(test_raw, 32)

print("Validation images:", val_count)
print("Training images after split:", train_size - val_count)

## 3. Quick look at the data

A small sample is enough here. There is no need to stare at thousands of cats and dogs to confirm that they are, in fact, cats and dogs.

In [ ]:
plt.figure(figsize=(10, 8))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8))
        plt.title(f"Class {labels[i].numpy()}")
        plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Helper functions

To avoid rewriting the same model code for every experiment, the model is created through a function.  
The MobileNetV2 base is initialized with ImageNet weights and a small classifier is added on top.

In [ ]:
def get_optimizer(name, lr):
    if name == "sgd":
        return keras.optimizers.SGD(learning_rate=lr)
    if name == "momentum":
        return keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    if name == "rmsprop":
        return keras.optimizers.RMSprop(learning_rate=lr)
    return keras.optimizers.Adam(learning_rate=lr)

def build_model(
    dropout_rate=0.25,
    l2_strength=0.0,
    use_batch_norm=False,
    optimizer="adam",
    learning_rate=1e-3,
    trainable_base=False
):
    base = keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = trainable_base

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = keras.applications.mobilenet_v2.preprocess_input(inputs)

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)

    if use_batch_norm:
        x = layers.BatchNormalization()(x)

    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        kernel_regularizer=regularizers.l2(l2_strength) if l2_strength > 0 else None
    )(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=get_optimizer(optimizer, learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def train_model(model, train_data, val_data, epochs=5, verbose=1):
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    ]
    start = time.time()
    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs,
        callbacks=callbacks,
        verbose=verbose
    )
    elapsed = time.time() - start
    return history, elapsed

def plot_histories(histories, metric="loss", title=""):
    plt.figure(figsize=(8, 5))
    for name, history in histories.items():
        plt.plot(history.history[metric], label=f"{name} train")
        val_metric = "val_" + metric
        if val_metric in history.history:
            plt.plot(history.history[val_metric], linestyle="--", label=f"{name} val")
    plt.xlabel("Epoch")
    plt.ylabel(metric.replace("_", " ").title())
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## 5. Weight initialization

MobileNetV2 itself already has pretrained weights, so replacing its initialization would not make much sense for this transfer-learning setup.  
Instead, the experiment compares different initializers in the newly added classifier layer.

In [ ]:
def build_initializer_model(initializer):
    base = keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu", kernel_initializer=initializer)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", kernel_initializer=initializer)(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

initializers = {
    "RandomNormal": keras.initializers.RandomNormal(stddev=0.05),
    "GlorotUniform": keras.initializers.GlorotUniform(),
    "HeNormal": keras.initializers.HeNormal()
}

init_histories = {}

for name, initializer in initializers.items():
    print(f"Running: {name}")
    model = build_initializer_model(initializer)
    history, _ = train_model(model, train_ds, val_ds, epochs=3)
    init_histories[name] = history

plt.figure(figsize=(8, 5))
for name, history in init_histories.items():
    plt.plot(history.history["loss"], label=name)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss for Different Initializers")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
for name, history in init_histories.items():
    plt.plot(history.history["val_accuracy"], label=name)
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy for Different Initializers")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Inference:** The curves show how the initialization of the classifier affects the first few stages of learning.  
The initializer that reaches a good validation accuracy with a smoother loss curve can be considered more stable for this setup.

## 6. Regularization and overfitting

Here the base model is kept frozen so the comparison mainly focuses on the classifier.  
The goal is to see whether L2 regularization, Dropout or Batch Normalization improves the gap between training and validation performance.

In [ ]:
regularization_configs = {
    "No regularization": dict(dropout_rate=0.0, l2_strength=0.0, use_batch_norm=False),
    "L2": dict(dropout_rate=0.0, l2_strength=1e-4, use_batch_norm=False),
    "Dropout": dict(dropout_rate=0.5, l2_strength=0.0, use_batch_norm=False),
    "BatchNorm": dict(dropout_rate=0.0, l2_strength=0.0, use_batch_norm=True)
}

reg_histories = {}

for name, config in regularization_configs.items():
    print(f"Running: {name}")
    model = build_model(**config, optimizer="adam", learning_rate=1e-3)
    history, _ = train_model(model, train_ds, val_ds, epochs=5)
    reg_histories[name] = history

plot_histories(reg_histories, "accuracy", "Training and Validation Accuracy")
plot_histories(reg_histories, "loss", "Training and Validation Loss")

## 7. Batch Normalization: small numerical example

The lab manual uses the values `[2, 4, 6, 8]`. This cell calculates the mean, variance and normalized values directly so the formula is easier to connect with actual numbers.

In [ ]:
x = np.array([2, 4, 6, 8], dtype=float)

mean = np.mean(x)
variance = np.mean((x - mean) ** 2)
x_hat = (x - mean) / np.sqrt(variance)

print("Input:", x)
print("Mean:", mean)
print("Variance:", variance)
print("Normalized values:", np.round(x_hat, 3))

## 8. Optimizer comparison

The same model is trained with SGD, Momentum, RMSProp and Adam.  
This keeps the comparison reasonably controlled instead of changing half the universe at once.

In [ ]:
optimizers = ["sgd", "momentum", "rmsprop", "adam"]
optimizer_histories = {}
optimizer_results = []

for opt in optimizers:
    print(f"Running: {opt}")
    model = build_model(
        dropout_rate=0.25,
        optimizer=opt,
        learning_rate=1e-3
    )
    history, elapsed = train_model(model, train_ds, val_ds, epochs=5)
    optimizer_histories[opt.upper()] = history

    best_val = max(history.history["val_accuracy"])
    final_loss = history.history["loss"][-1]

    optimizer_results.append({
        "Optimizer": opt.upper(),
        "Final Loss": final_loss,
        "Best Validation Accuracy": best_val,
        "Epochs Run": len(history.history["loss"]),
        "Time (s)": elapsed
    })

plt.figure(figsize=(8, 5))
for name, history in optimizer_histories.items():
    plt.plot(history.history["loss"], label=name)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss for Different Optimizers")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
for name, history in optimizer_histories.items():
    plt.plot(history.history["val_accuracy"], label=name)
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy for Different Optimizers")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

optimizer_df = pd.DataFrame(optimizer_results)
display(optimizer_df)

## 9. Hyperparameter tuning

A full grid search would take a long time on a normal Colab runtime, so this notebook changes one setting at a time.  
The values follow the ranges suggested in the experiment.

In [ ]:
# Learning rate study
learning_rates = [1e-3, 1e-4]
lr_results = []

for lr in learning_rates:
    model = build_model(learning_rate=lr, optimizer="adam")
    history, _ = train_model(model, train_ds, val_ds, epochs=3, verbose=0)
    lr_results.append(max(history.history["val_accuracy"]))

plt.figure(figsize=(6, 4))
plt.plot(learning_rates, lr_results, marker="o")
plt.xscale("log")
plt.xlabel("Learning Rate")
plt.ylabel("Best Validation Accuracy")
plt.title("Learning Rate vs Validation Accuracy")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Batch size study
batch_sizes = [16, 32, 64]
batch_results = []

for batch_size in batch_sizes:
    train_b = make_dataset(train_raw, batch_size, training=True)
    val_b = make_dataset(val_raw, batch_size)

    model = build_model(learning_rate=1e-3, optimizer="adam")
    history, _ = train_model(model, train_b, val_b, epochs=3, verbose=0)
    batch_results.append(max(history.history["val_accuracy"]))

plt.figure(figsize=(6, 4))
plt.plot(batch_sizes, batch_results, marker="o")
plt.xlabel("Batch Size")
plt.ylabel("Best Validation Accuracy")
plt.title("Batch Size vs Validation Accuracy")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Dropout study
dropout_rates = [0.0, 0.25, 0.5]
dropout_results = []

for rate in dropout_rates:
    model = build_model(
        dropout_rate=rate,
        learning_rate=1e-3,
        optimizer="adam"
    )
    history, _ = train_model(model, train_ds, val_ds, epochs=3, verbose=0)
    dropout_results.append(max(history.history["val_accuracy"]))

plt.figure(figsize=(6, 4))
plt.plot(dropout_rates, dropout_results, marker="o")
plt.xlabel("Dropout Rate")
plt.ylabel("Best Validation Accuracy")
plt.title("Dropout Rate vs Validation Accuracy")
plt.grid(True, alpha=0.3)
plt.show()

## 10. Feature extraction and fine-tuning

First, the MobileNetV2 base is frozen and only the classifier is trained.  
After that, the upper part of the base network is unfrozen and trained with a much smaller learning rate.

In [ ]:
feature_model = build_model(
    dropout_rate=0.25,
    optimizer="adam",
    learning_rate=1e-3,
    trainable_base=False
)

feature_history, feature_time = train_model(
    feature_model, train_ds, val_ds, epochs=5
)

# Unfreeze only the upper layers
base_model = next(
    layer for layer in feature_model.layers
    if isinstance(layer, keras.Model) and "mobilenetv2" in layer.name.lower()
)

base_model.trainable = True

# Freeze most layers and fine-tune only the later part
for layer in base_model.layers[:-30]:
    layer.trainable = False

feature_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fine_history, fine_time = train_model(
    feature_model, train_ds, val_ds, epochs=5
)

plt.figure(figsize=(8, 5))
plt.plot(feature_history.history["val_accuracy"], label="Feature Extraction")
plt.plot(
    range(len(feature_history.history["val_accuracy"]),
          len(feature_history.history["val_accuracy"]) + len(fine_history.history["val_accuracy"])),
    fine_history.history["val_accuracy"],
    label="Fine-Tuning"
)
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Feature Extraction vs Fine-Tuning")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Inference:** Fine-tuning starts from an already trained representation, so a smaller learning rate is used to avoid changing useful pretrained features too aggressively.

## 11. Small 5-fold cross-validation study

Cross-validation on the full Oxford-IIIT Pet dataset can be expensive in Colab.  
To keep the notebook practical, a manageable subset is used here. The purpose is still the same: compare promising configurations using both mean accuracy and variability.

In [ ]:
# Convert a subset of the training data to NumPy arrays for StratifiedKFold
CV_SAMPLES = 500

cv_images = []
cv_labels = []

for image, label in train_raw.take(CV_SAMPLES):
    cv_images.append(image.numpy())
    cv_labels.append(label.numpy())

cv_images = np.array(cv_images)
cv_labels = np.array(cv_labels)

configs = {
    "C1": {"dropout_rate": 0.25, "optimizer": "adam", "learning_rate": 1e-3},
    "C2": {"dropout_rate": 0.5, "optimizer": "adam", "learning_rate": 1e-3},
    "C3": {"dropout_rate": 0.25, "optimizer": "sgd", "learning_rate": 1e-3}
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_summary = []

for config_name, config in configs.items():
    fold_scores = []

    print(f"\nConfiguration: {config_name}")

    for fold, (train_idx, val_idx) in enumerate(skf.split(cv_images, cv_labels), start=1):
        fold_train = tf.data.Dataset.from_tensor_slices(
            (cv_images[train_idx], cv_labels[train_idx])
        )
        fold_val = tf.data.Dataset.from_tensor_slices(
            (cv_images[val_idx], cv_labels[val_idx])
        )

        fold_train = make_dataset(fold_train, 32, training=True)
        fold_val = make_dataset(fold_val, 32)

        model = build_model(**config)
        history, _ = train_model(
            model, fold_train, fold_val, epochs=2, verbose=0
        )

        score = max(history.history["val_accuracy"])
        fold_scores.append(score)
        print(f"Fold {fold}: {score:.4f}")

    cv_summary.append({
        "Configuration": config_name,
        "F1": fold_scores[0],
        "F2": fold_scores[1],
        "F3": fold_scores[2],
        "F4": fold_scores[3],
        "F5": fold_scores[4],
        "Mean": np.mean(fold_scores),
        "SD": np.std(fold_scores)
    })

cv_df = pd.DataFrame(cv_summary)
display(cv_df)

In [ ]:
plt.figure(figsize=(7, 5))
plt.errorbar(
    cv_df["Configuration"],
    cv_df["Mean"],
    yerr=cv_df["SD"],
    fmt="o",
    capsize=5
)
plt.xlabel("Hyperparameter Configuration")
plt.ylabel("Mean Validation Accuracy")
plt.title("5-Fold Cross-Validation Accuracy")
plt.grid(True, alpha=0.3)
plt.show()

## 12. Final model evaluation

The best configuration from cross-validation is selected using the highest mean accuracy, while also checking the standard deviation.  
The untouched test set is only used at this stage.

In [ ]:
best_row = cv_df.sort_values("Mean", ascending=False).iloc[0]
best_name = best_row["Configuration"]
best_config = configs[best_name]

print("Selected configuration:", best_name)
print(best_config)

final_model = build_model(**best_config)
final_history, final_time = train_model(
    final_model, train_ds, val_ds, epochs=8
)

test_loss, test_accuracy = final_model.evaluate(test_ds, verbose=1)

y_true, y_pred = [], []

for images, labels in test_ds:
    predictions = final_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"Training Time: {final_time:.2f} seconds")
print(f"Parameters: {final_model.count_params():,}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12, 10))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.colorbar()
plt.show()

## 13. Overall results

The table below brings together the main experiments. The exact values will depend on the Colab runtime, random seed and number of epochs, so the output generated during execution should be used in the final lab record.

In [ ]:
overall_results = pd.DataFrame({
    "Configuration": [
        "Baseline",
        "Best Initializer",
        "Best Regularization",
        "Best Optimizer",
        "Best Hyperparameters",
        "Fine-Tuned Model"
    ],
    "CV Accuracy": ["-", "-", "-", "-", best_row["Mean"], "-"],
    "SD": ["-", "-", "-", "-", best_row["SD"], "-"],
    "Test Accuracy": ["-", "-", "-", "-", test_accuracy, test_accuracy],
    "Training Time": ["-", "-", "-", "-", final_time, fine_time]
})

display(overall_results)

## 14. Final observation

The experiment shows that model performance is not decided by architecture alone. Initialization affects early learning, regularization influences generalization, optimizers change the training behaviour, and fine-tuning can improve a pretrained model when it is done carefully.

The final configuration should be chosen by considering validation performance, cross-validation variability, training cost and final test accuracy rather than simply picking the single highest number.